# Limpieza de fuentes — los 7 archivos de la rama


## Datasets gestionados
| Dataset | Original (`01_raw`) | Limpio (`02_interim`) | Rol |
|---|---|---|---|
| JobHop v2 `train` | `JobHop_v2_train.csv` | `JobHop_v2_train_limpio.csv` | Base de trayectorias (quién, cuándo, qué ocupación) |
| JobHop v2 `test` | `JobHop_v2_test.csv` | `JobHop_v2_test_limpio.csv` | Ídem, partición de prueba |
| JobHop v2 `val` | `JobHop_v2_val.csv` | `JobHop_v2_val_limpio.csv` | Ídem, partición de validación |
| ESCO occupations | `ESCO/occupations_en.csv` | `ESCO/occupations_en_limpio.csv` | Traduce `code` → nombre de ocupación + grupo ISCO-08 |
| ESCO ISCOGroups | `ESCO/ISCOGroups_en.csv` | `ESCO/ISCOGroups_en_limpio.csv` | Etiqueta y jerarquía del área ocupacional (`code` 4d → label + nivel) |
| ESCO skills | `SKILLS/skills_en.csv` | `SKILLS/skills_en_limpio.csv` | Ficha de cada competencia (texto, tipo) |
| ESCO relations | `SKILLS/occupationSkillRelations_en.csv` | `SKILLS/occupationSkillRelations_en_limpio.csv` | Puente oficio → competencias (base de `n_skills_*`) |

`limpiar_datos.py` recorre `data/01_raw/` con `rglob` y limpia **todo** lo que
encuentra salvo el propio directorio `03_processed`, conservando la subcarpeta
de origen. Por eso `ESCO/` y `SKILLS/` aparecen como carpetas hermanas en
`02_interim/`: no se renombran ni se aplanan.

**Ojo con la rama:** la integración usa **test + val** (376.567 experiencias), no
`train`. `train` se limpia igual, pero su versión limpia no entra al integrado.

## Preparación

Localiza la raíz del proyecto y reutiliza las funciones de `limpiar_datos.py` para que este cuaderno haga **exactamente** lo mismo que el script.

In [1]:
import sys
from pathlib import Path
import pandas as pd
from IPython.display import display
PROYECTO = Path.cwd()
while not (PROYECTO / "data").exists() and PROYECTO != PROYECTO.parent:
    PROYECTO = PROYECTO.parent
sys.path.insert(0, str(PROYECTO / "src" / "limpieza"))
import limpiar_datos as L
print("Raíz del proyecto:", PROYECTO)
print("Origen :", L.ORIGEN)
print("Destino:", L.DESTINO)

Raíz del proyecto: C:\Users\ACER\Desktop\MineriaProtect
Origen : C:\Users\ACER\Desktop\MineriaProtect\data\01_raw
Destino: C:\Users\ACER\Desktop\MineriaProtect\data\02_interim


## Qué se procesa (detección automática)
`limpiar_datos.py` recorre `data/01_raw/` con `rglob("*")` y limpia cada
`csv`/`parquet` que encuentre. La subcarpeta se conserva: un archivo en
`01_raw/SKILLS/` sale en `02_interim/SKILLS/`.

In [2]:
archivos = [
    p
    for p in sorted(L.ORIGEN.rglob("*"))
    if p.is_file() and p.suffix.lower() in (L.EXT_CSV, L.EXT_PARQUET)
]
display(pd.DataFrame({"archivo": [str(p.relative_to(L.ORIGEN)) for p in archivos]}))
print("Total a procesar:", len(archivos))

,archivo
0,ESCO\ISCOGroups_en.csv
1,ESCO\occupations_en.csv
2,JobHop_v2_test.csv
3,JobHop_v2_train.csv
4,JobHop_v2_val.csv
5,SKILLS\occupationSkillRelations_en.csv
6,SKILLS\skills_en.csv


Total a procesar: 7


## Procesamiento (mismos pasos y validación que `limpiar_datos.py`)
La función `procesar` aplica: inspección → limpieza (`limpiar_csv`) → validación → guardado en `data/02_interim/`.

In [ ]:
def procesar(ruta: Path):
    rel = str(ruta.relative_to(L.ORIGEN)).replace("\\", "/")
    df = L.carga_archivo(ruta)
    L.inspeccionar(
        df, rel, "Parquet" if ruta.suffix.lower() == L.EXT_PARQUET else "CSV"
    )

    cambios = []
    stats = {
        "filas_eliminadas": False,
        "columnas_eliminadas": False,
        "valores_modificados": False,
        "cambio_tipos": False,
        "texto_normalizado": False,
        "duplicados_eliminados": False,
    }
    df_limpio = L.limpiar_csv(df, ruta.stem, cambios, stats)

    validacion = {
        "filas": len(df),
        "columnas": df.shape[1],
        "dups_fila": int(df.duplicated().sum()),
        "nulos_criticos": L.nulos_criticos(df, ruta.stem),
    }
    L.validar(rel, validacion, df_limpio, cambios)

    destino = L.destino_para(ruta.relative_to(L.ORIGEN))
    destino.parent.mkdir(parents=True, exist_ok=True)
    df_limpio.to_csv(destino, index=False, encoding="utf-8")
    print("Guardado en:", destino)
    return df_limpio

### occupations_en.csv

3.043 originales → 3.039: **−4 filas** por `code` duplicado (idénticas salvo `modifiedDate`).

In [ ]:
df_occ = procesar(next(p for p in archivos if p.stem == "occupations_en"))

### ISCOGroups_en.csv

619 originales → 619: se elimina la columna `altLabels` (100 % nula): de 8 a 7 columnas.

In [ ]:
df_isco = procesar(next(p for p in archivos if p.stem == "ISCOGroups_en"))

## JobHop v2 — verificar la versión limpia
Las tres particiones (`train`/`test`/`val`) **sí se limpian aquí**, con el mismo
tratamiento que las fuentes ESCO. La integración posterior solo consume
`test` + `val`.

In [3]:
for nombre in ("train", "test", "val"):
    ruta = L.DESTINO / f"JobHop_v2_{nombre}_limpio.csv"
    df = L.carga_archivo(ruta)
    print(f"{ruta.name:<34} {len(df):>9,} filas x {df.shape[1]} columnas")
    if nombre == "val":
        display(df.head(3))

JobHop_v2_train_limpio.csv         1,506,445 filas x 5 columnas
JobHop_v2_test_limpio.csv            188,061 filas x 5 columnas


JobHop_v2_val_limpio.csv             188,506 filas x 5 columnas


,resume_id,matched_code,start_date,end_date,university_level
0,10,1222.1.2,Q1 2011,Q1 2015,Master
1,10,2310.1.10,Q1 2014,Q1 2015,Master
2,10,2431.16,Q1 2015,Q3 2017,Master


## Cierre
- Salidas: `data/02_interim/**/*_limpio.csv` (este cuaderno), con la subcarpeta
  de origen conservada (`ESCO/` y `SKILLS/`).
- Los originales de `data/01_raw/` **no se modifican**: solo se leen.
- El proceso es **idempotente**: re-ejecutarlo produce los mismos archivos
  (validación antes/después por archivo).
- Siguiente etapa: `integrar_empleos.py` produce el integrado
  `data/03_processed/empleos.csv`; después `limpiar_empleos.py` y
  `dividir_por_outliers.py`.